# 08c -- KeepTradeCut Rankings

**Purpose**: Scrape KeepTradeCut dynasty rookie rankings and ingest into
`fact_rookie_rankings`. Self-contained.

**Sources**: Both value sets are extracted from a **single HTTP fetch** (both embedded
in `var playersArray` JSON in the page HTML — no auth required).

| Format scraped | `values_key` | notes |
|---|---|---|
| 1QB | `oneQBValues` | base |
| Superflex | `superflexValues` | base |
| SF TE++ | `superflexValues.tepp` | nested sub-key |

**`grade`**: KTC trade value (0–10 000 crowdsourced scale — not a 0–100 scouting grade).

**Aggregation**: All three formats are averaged per player into a single `KTC_Consensus`
row — `global_rank`, `positional_rank`, and `grade` (trade value) are each the mean across
the three format variants. One HTTP fetch yields all three (sub-keys nested in the same JSON).

**`rank_date`**: set to `TODAY` — KTC updates continuously, no fixed publish date.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    review_dir: str = "data/review"          # all review CSVs live here
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG    = LeagueConfig()
DATA   = Path(CFG.data_dir)
REVIEW = Path(CFG.review_dir)
TODAY  = date.today().isoformat()
DATA.mkdir(exist_ok=True)
REVIEW.mkdir(parents=True, exist_ok=True)
ALIAS  = DATA / "dim_player_alias.parquet"   # persistent name->player_key decisions (08y/08z)


def _make_session(timeout_sec: int = 30, retries: int = 3, backoff: float = 2.0) -> requests.Session:
    # Shared session factory: retry on timeout/5xx with exponential backoff.
    # backoff waits: 2s, 4s, 8s between attempts.
    session = requests.Session()
    retry = Retry(
        total=retries,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    return session

In [2]:
# -- Player name/key helpers (copied from 01_dim_rookie_prospect) -------------

def clean_player_name(name: str) -> str:
    # Normalize for matching: remove periods, collapse whitespace, lowercase.
    if pd.isna(name):
        return ""
    s = str(name).strip()
    s = s.replace(".", "").replace("\u00a0", " ")
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("`", "'")
    return " ".join(s.split()).lower()


def generate_player_key(name: str, position: str, school: str) -> str:
    # Deterministic 12-char MD5 hash -- matches keys generated in 01.
    raw = f"{clean_player_name(name)}|{str(position).upper().strip()}|{str(school).strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


assert clean_player_name("J.K. Dobbins") == "jk dobbins"
assert clean_player_name("D'Andre Swift") == "d'andre swift"
print("Helpers OK")

def _parse_rank_date(raw: str | None) -> str | None:
    # Parse the source-published "last updated" date into ISO format.
    # Returns None if raw is empty or unparseable (caller stores NULL).
    if not raw:
        return None
    raw = str(raw).strip()
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(raw, fmt).date().isoformat()
        except ValueError:
            continue
    return raw  # store as-is if no format matched


Helpers OK


## 2 -- add_players_from_source

In [ ]:
def add_players_from_source(
    new_players_df: pd.DataFrame,
    source_name: str,
    draft_year: int = CFG.draft_year,
    auto_threshold: int = CFG.fuzzy_auto_threshold,
    review_threshold: int = CFG.fuzzy_review_threshold,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Fuzzy-match new player names against dim_rookie_prospect.
    # Returns (updated_dim_rookie_prospect, review_df).
    # new_players_df must have: player_name; optionally position_raw, school_raw.
    # NOTE: review_df is returned to caller -- file write is the caller's responsibility.
    dim_rp      = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    pos_map     = pd.read_parquet(DATA / "dim_position.parquet")
    school_map  = pd.read_parquet(DATA / "dim_school.parquet")

    existing_names  = dim_rp["player_name_clean"].tolist()
    existing_lookup = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))

    # Persistent decisions: (name_clean, position_raw) already resolved in a prior
    # review -> never ask again (dim_player_alias, built by 08y, appended by 08z).
    alias_keys = set()
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        alias_keys = set(zip(_al["name_clean"], _al["position_raw"]))

    new_rows, review_rows = [], []
    auto_matched = already_exists = alias_resolved = 0

    for _, row in new_players_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pos_key    = str(row.get("position_raw", "")).upper().strip()

        if name_clean in existing_lookup:
            already_exists += 1
            continue

        # Already decided in a past review -> skip silently (no repeat review).
        if (name_clean, pos_key) in alias_keys:
            alias_resolved += 1
            continue

        best_match, score = ("", 0)
        if existing_names:
            best_match, score = process.extractOne(
                name_clean, existing_names, scorer=fuzz.token_sort_ratio
            )

        if score >= auto_threshold:
            auto_matched += 1
        elif score >= review_threshold:
            review_rows.append({
                "new_name":        row["player_name"],
                "new_name_clean":  name_clean,
                "new_position":    row.get("position_raw", ""),
                "new_school":      row.get("school_raw", ""),
                "best_match_name": best_match,
                "best_match_key":  existing_lookup.get(best_match, ""),
                "fuzzy_score":     score,
                "action":          "",  # fill: "match" or "new"
                "source":          source_name,
            })
        else:
            # New prospect -- add to dim_rookie_prospect
            pos_raw    = str(row.get("position_raw", "")).upper().strip()
            school_raw = str(row.get("school_raw", "")).strip()
            pos_match  = pos_map[pos_map["position_raw"] == pos_raw]
            sch_match  = school_map[school_map["school_raw"] == school_raw]
            pkey       = generate_player_key(row["player_name"], pos_raw, school_raw)
            new_rows.append({
                "player_key":       pkey,
                "player_name":      row["player_name"],
                "player_name_clean": name_clean,
                "position_raw":     pos_raw,
                "position_detail":  pos_match["position_detail"].iloc[0] if len(pos_match) else None,
                "position_group":   pos_match["position_group"].iloc[0]  if len(pos_match) else None,
                "side_of_ball":     pos_match["side_of_ball"].iloc[0]    if len(pos_match) else None,
                "fantasy_relevant": pos_match["fantasy_relevant"].iloc[0] if len(pos_match) else False,
                "school_raw":       school_raw,
                "school_canonical": sch_match["school_canonical"].iloc[0] if len(sch_match) else school_raw,
                "conference":       sch_match["conference"].iloc[0]       if len(sch_match) else "Unknown",
                "height_inches":    None, "weight": None,
                "pfr_id":           None, "cfb_id": None,
                "gsis_id":          pd.NA,
                "draft_year":       draft_year,
                "source":           source_name,
                "added_date":       TODAY,
            })
            existing_names.append(name_clean)
            existing_lookup[name_clean] = pkey

    if new_rows:
        dim_rp = pd.concat([dim_rp, pd.DataFrame(new_rows)], ignore_index=True)
        dim_rp.drop_duplicates(subset=["player_key"], inplace=True)
        dim_rp.to_parquet(DATA / "dim_rookie_prospect.parquet", index=False)

    review_df = pd.DataFrame(review_rows) if review_rows else pd.DataFrame()

    print(f"Source: {source_name}")
    print(f"  Already in dim_rookie_prospect : {already_exists}")
    print(f"  Resolved via alias (no re-ask)  : {alias_resolved}")
    print(f"  Auto-matched (>={auto_threshold})       : {auto_matched}")
    print(f"  New prospects added             : {len(new_rows)}")
    print(f"  Needs manual review             : {len(review_rows)}")

    return dim_rp, review_df

## 3 -- ingest_ranking_source

In [ ]:
def ingest_ranking_source(
    rankings_df: pd.DataFrame,
    source_name: str,
    source_site: str,
    phase: str,
    draft_year: int = CFG.draft_year,
    capture_date: str = TODAY,
    rank_date: str | None = None,
) -> pd.DataFrame:
    # Append one row per player to fact_rookie_rankings for a given source + phase.
    # Call add_players_from_source() first to ensure all players are in dim_rookie_prospect.
    # Players pending review (not yet in dim_rookie_prospect) are logged as unmatched and skipped.
    #
    # rankings_df required columns: player_name, global_rank
    # rankings_df optional columns: positional_rank, grade

    valid_phases = {"pre_combine", "post_combine", "post_draft"}
    if phase not in valid_phases:
        raise ValueError(f"phase must be one of {valid_phases}")

    # name -> player_key lookup from dim_rookie_prospect
    dim_rp = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    name_to_key = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))
    key_to_pfr  = dict(zip(dim_rp["player_key"],        dim_rp["pfr_id"]))

    # Fold in alias decisions so a matched name-variant (X resolved to prospect Y)
    # attributes its ranking to Y's player_key instead of being dropped as unmatched.
    # setdefault: real dim_rookie_prospect names always win over an alias.
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        for _nc, _pk in zip(_al["name_clean"], _al["player_key"]):
            name_to_key.setdefault(_nc, _pk)

    # pfr_id -> gsis_id lookup from dim_nfl_players (null pre-signing)
    dim_nfl = pd.read_parquet(DATA / "dim_nfl_players.parquet")
    pfr_to_gsis = dict(zip(dim_nfl["pfr_id"].dropna(), dim_nfl["gsis_id"].dropna()))

    rows, unmatched = [], []
    for _, row in rankings_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pkey = name_to_key.get(name_clean)

        if pkey is None:
            unmatched.append(row["player_name"])
            continue

        pfr_id  = key_to_pfr.get(pkey)
        gsis_id = pfr_to_gsis.get(pfr_id) if pfr_id else None

        rows.append({
            "player_key":      pkey,
            "gsis_id":         gsis_id,
            "source_name":     source_name,
            "source_site":     source_site,
            "phase":           phase,
            "draft_year":      draft_year,
            "global_rank":     row.get("global_rank"),
            "positional_rank": row.get("positional_rank"),
            "grade":           row.get("grade"),
            "capture_date":    capture_date,
            "rank_date":       rank_date,
        })

    if unmatched:
        print(f"WARN: {len(unmatched)} players pending review -- will be picked up after apply_review_decisions():")
        for name in unmatched[:10]:
            print(f"  {name}")
        if len(unmatched) > 10:
            print(f"  ... and {len(unmatched) - 10} more")

    new_df = pd.DataFrame(rows)
    if new_df.empty:
        print("No rows to append.")
        return new_df

    new_df["global_rank"]     = new_df["global_rank"].astype("Int64")
    new_df["positional_rank"] = new_df["positional_rank"].astype("Int64")
    new_df["draft_year"]      = new_df["draft_year"].astype("Int64")
    new_df["rank_date"]       = new_df["rank_date"].where(new_df["rank_date"].notna(), other=None)

    existing = pd.read_parquet(DATA / "fact_rookie_rankings.parquet")
    combined = pd.concat([existing, new_df], ignore_index=True)
    _DEDUP   = ["player_key", "source_name", "phase", "draft_year"]
    # rank_date: preserve the first non-null value ever captured — never overwrite with a later scrape
    combined["rank_date"] = (
        combined.groupby(_DEDUP, sort=False)["rank_date"]
        .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
    )
    combined.drop_duplicates(subset=_DEDUP, keep="last", inplace=True)
    combined.to_parquet(DATA / "fact_rookie_rankings.parquet", index=False)

    print(f"Ingested: {len(rows)} rows | source={source_name} | site={source_site} | phase={phase}")
    print(f"fact_rookie_rankings total: {len(combined)}")
    return new_df

## 4 -- KeepTradeCutScraper

`var playersArray` JSON is embedded directly in the HTML — no API key, no auth.
A single `requests.get()` returns all 57 rookies with both `superflexValues` and
`oneQBValues`. The scraper caches the response internally so both sources share one fetch.

In [5]:
class KeepTradeCutScraper:
    # Parses var playersArray JSON embedded in KTC dynasty rookie rankings page.
    # Both superflexValues and oneQBValues are present in every response, so a
    # single HTTP fetch yields two source rows per player.
    #
    # grade field stores the raw KTC trade value (0-10000 crowdsourced scale).
    # rank_date = TODAY because KTC rankings update continuously.

    _URL = "https://keeptradecut.com/dynasty-rankings/rookie-rankings"
    _PLAYERS_PAT = re.compile(r"var playersArray = (\[.*?\]);", re.DOTALL)

    _HEADERS = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    SOURCES = {
        "KTC_1QB": {
            "values_key":     "oneQBValues",
            "site_reference": "Dynasty Rookie Rankings - 1QB",
            "source_site":    "KeepTradeCut",
        },
        "KTC_Superflex": {
            "values_key":     "superflexValues",
            "site_reference": "Dynasty Rookie Rankings - Superflex",
            "source_site":    "KeepTradeCut",
        },
        "KTC_TEpp": {
            "values_key":     "superflexValues",
            "sub_key":        "tepp",          # nested: superflexValues.tepp
            "site_reference": "Dynasty Rookie Rankings - SF TE++",
            "source_site":    "KeepTradeCut",
        },
    }

    def __init__(self, timeout: int = 30, delay: float = 2.0):
        self.timeout  = timeout
        self.delay    = delay
        self.session  = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)
        self._cached_players: list | None = None  # reuse across scrape() calls

    def _fetch_players(self) -> list:
        # Fetch page once; cache for subsequent calls in the same session.
        if self._cached_players is not None:
            return self._cached_players
        time.sleep(self.delay)
        resp = self.session.get(self._URL, timeout=self.timeout)
        resp.raise_for_status()
        m = self._PLAYERS_PAT.search(resp.text)
        if not m:
            raise ValueError("var playersArray not found in KTC page HTML")
        self._cached_players = json.loads(m.group(1))
        return self._cached_players

    def scrape(self, source_key: str) -> tuple[pd.DataFrame, str | None]:
        # Returns (df, rank_date). rank_date = TODAY — KTC updates continuously.
        cfg     = self.SOURCES[source_key]
        vkey    = cfg["values_key"]
        sub_key = cfg.get("sub_key")      # optional nested key (e.g., "tepp")
        players = self._fetch_players()

        rows = []
        for i, p in enumerate(players):
            vals = p.get(vkey, {})
            if sub_key:
                vals = vals.get(sub_key, {})  # drill into nested structure
            if not vals:
                continue
            rows.append({
                "player_name":     p["playerName"],
                "position_raw":    p["position"],
                "school_raw":      p.get("college", ""),
                "global_rank":     vals.get("rookieRank"),
                "positional_rank": vals.get("rookiePositionalRank"),
                "grade":           float(vals["value"]) if vals.get("value") is not None else None,
                "ktc_value":       vals.get("value"),
                "ktc_tier":        vals.get("rookieTier"),
                "trend_30d":       vals.get("overallTrend"),
                "nfl_team":        p.get("team", ""),
                "ktc_player_id":   p.get("playerID"),
            })

        if not rows:
            raise ValueError(f"No player rows parsed for {source_key}")

        df = pd.DataFrame(rows)
        rank_date = TODAY  # KTC is continuously updated; use capture date as rank_date
        print(f"Scraped {source_key}: {len(df)} players | {cfg['site_reference']} | rank_date={rank_date}")
        return df, rank_date

## 5 -- Run KeepTradeCut Ingestion

In [6]:
SOURCES = {"KTC_Consensus": {"source_site": "KeepTradeCut"}}  # alias for validation cell

KTC_SCRAPER = KeepTradeCutScraper(timeout=30, delay=2.0)
PHASE       = "post_draft"
_failed     = []
_all_reviews= []
_scraped    = {}  # source_key -> (df, rank_date)

# ── Step 1: Scrape 3 formats — single HTTP fetch via internal cache ────────────
print("Scraping KTC formats (1 fetch, 3 sub-keys)...")
for source_key in KeepTradeCutScraper.SOURCES:
    cfg = KeepTradeCutScraper.SOURCES[source_key]
    try:
        rankings_df, rank_date = KTC_SCRAPER.scrape(source_key)
        _scraped[source_key] = (rankings_df, rank_date)
    except Exception as e:
        print(f"  ERROR {source_key}: {e}")
        _failed.append(source_key)

if not _scraped:
    print("No sources scraped — aborting.")
else:
    # ── Step 2: add_players_from_source once (players identical across formats) ──
    _first_key = next(iter(_scraped))
    _first_df, _ = _scraped[_first_key]
    try:
        _, review = add_players_from_source(
            _first_df, source_name="KTC", draft_year=CFG.draft_year
        )
        if not review.empty:
            print(f"  -> {len(review)} players flagged for review")
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players: {e}")

    # ── Step 3: Aggregate 3 formats -> one consensus row per player ────────────
    # global_rank, positional_rank, and grade (trade value) are each averaged
    # across 1QB / Superflex / TE++.
    _all_dfs = []
    for source_key, (df, rank_date) in _scraped.items():
        _d = df.copy()
        _d["_format"]     = source_key
        _d["_rank_date"]  = rank_date
        _all_dfs.append(_d)

    _combined = pd.concat(_all_dfs, ignore_index=True)
    _combined["_name_clean"] = _combined["player_name"].apply(clean_player_name)

    agg_ktc = (
        _combined.groupby("_name_clean", dropna=False)
        .agg(
            player_name     = ("player_name",     "first"),
            position_raw    = ("position_raw",    "first"),
            school_raw      = ("school_raw",      "first"),
            global_rank     = ("global_rank",     "mean"),
            positional_rank = ("positional_rank", "mean"),
            grade           = ("grade",           "mean"),   # avg trade value (0-10000 scale)
            rank_date       = ("_rank_date",      "max"),
            _formats        = ("_format",         lambda x: "+".join(sorted(set(x)))),
        )
        .reset_index(drop=True)
    )
    agg_ktc["global_rank"]     = agg_ktc["global_rank"].round().astype("Int64")
    agg_ktc["positional_rank"] = agg_ktc["positional_rank"].round().astype("Int64")

    n_formats = len(_scraped)
    print(f"\n{n_formats} formats x {len(_combined)//n_formats} players "
          f"-> {len(agg_ktc)} KTC_Consensus rows")
    print("\nTop 10 consensus:")
    print(agg_ktc.sort_values("global_rank")[
        ["player_name","position_raw","global_rank","positional_rank","grade"]
    ].head(10).to_string(index=False))

    # ── Step 4: Ingest KTC_Consensus ──────────────────────────────────────────
    try:
        ingest_ranking_source(
            agg_ktc,
            source_name="KTC_Consensus",
            source_site="KeepTradeCut",
            phase=PHASE,
            draft_year=CFG.draft_year,
            rank_date=agg_ktc["rank_date"].max(),
        )
    except Exception as e:
        print(f"  ERROR ingesting KTC_Consensus: {e}")
        _failed.append("KTC_Consensus")

    # ── Review file (append-safe across notebooks) ─────────────────────────────
    if _all_reviews:
        new_review = pd.concat(_all_reviews, ignore_index=True)
        rp = DATA / "review" / "review_fuzzy_matches.csv"
        if rp.exists():
            existing = pd.read_csv(rp)
            combined = pd.concat([existing, new_review], ignore_index=True)
            # keep="first" preserves already-filled action values from prior review sessions
            combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
            n_added = len(combined) - len(existing)
            combined.to_csv(rp, index=False)
            print("\n" + "=" * 60)
            print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
        else:
            new_review.to_csv(rp, index=False)
            print("\n" + "=" * 60)
            print(f"Review file: {rp}  ({len(new_review)} rows)")
        print("Fill \"action\" column (\"match\" or \"new\"), run 09_apply_fuzzy_review, then re-run.")

    print("\n" + "=" * 60)
    if _failed:
        print(f"Failed: {_failed}")
    else:
        print("KTC_Consensus ingested.")

Scraping KTC formats (1 fetch, 3 sub-keys)...
Scraped KTC_1QB: 57 players | Dynasty Rookie Rankings - 1QB | rank_date=2026-05-16
Scraped KTC_Superflex: 57 players | Dynasty Rookie Rankings - Superflex | rank_date=2026-05-16
Scraped KTC_TEpp: 57 players | Dynasty Rookie Rankings - SF TE++ | rank_date=2026-05-16
Source: KTC
  Already in dim_rookie_prospect : 54
  Auto-matched (>=90)       : 0
  New prospects added             : 0
  Needs manual review             : 3
  -> 3 players flagged for review

3 formats x 57 players -> 57 KTC_Consensus rows

Top 10 consensus:
     player_name position_raw  global_rank  positional_rank       grade
  Jeremiyah Love           RB            1                1 7775.333333
    Carnell Tate           WR            2                1 6060.666667
    Jordyn Tyson           WR            4                2 5523.666667
Fernando Mendoza           QB            5                1 5336.666667
     Makai Lemon           WR            5                3 5296.666

## 6 -- Validation

In [7]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = pd.read_parquet(Path(CFG.data_dir) / "fact_rookie_rankings.parquet")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 54
source_name    phase     
KTC_Consensus  post_draft    54

fact_rookie_rankings total: 363 rows
Composite key duplicates: 0 (expected 0)
